##### Import statements:

In [100]:
import numpy as np
import matplotlib.pyplot
import torch
import torch.nn as nn
import torch.nn.functional as F 
from torch.autograd import Variable
from torch.utils.data import DataLoader
import time

##### Define parameters:

In [109]:
# Data parameters:
n_feat = 80
n_obs = 320
n_classes = 2
data_noise = 0.1

# Model parameters:
n_inp = n_feat
n_hidden = 100
sigma_init = 1
sigma_noise = 0.1
batch_size = 64
n_epochs = 500
lr = 0.001
beta_ce = 1
beta_sp = 1
p_norm = 2

##### Define model class:

In [103]:
class Mdl(nn.Module):
    def __init__(self,n_inp,n_hidden, n_classes, sigma_init):    
        super(Mdl,self).__init__()
        self.n_inp=n_inp
        self.n_hidden=n_hidden
        self.n_classes=n_classes
        self.sigma_init=sigma_init
        self.enc=torch.nn.Linear(n_inp,n_hidden)
        self.dec=torch.nn.Linear(n_hidden,n_classes)
        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=self.sigma_init)
            if module.bias is not None:
                module.bias.data.normal_(mean=0.0, std=self.sigma_init)

    def forward(self,x,sigma_noise,gpu=False):
        if not gpu:
            x_hidden = F.relu(self.enc(x))+sigma_noise*torch.randn(x.size(0),self.n_hidden)
        else:
            x_hidden = F.relu(self.enc(x))+sigma_noise*torch.randn(x.size(0),self.n_hidden).to('cuda')
        x = self.dec(x_hidden)
        return x,x_hidden


def sparsity_loss(data,p):
    loss=torch.mean(torch.pow(abs(data),p),axis=(0,1))
    return loss

##### Generate data:

In [104]:
# Define centroid for each class:
mus = []
for i in np.arange(n_classes):
    curr_mu = np.random.randn(n_feat).astype(np.float32)
    mus.append(curr_mu)

# Define covariance matrix:
C = np.random.randn(n_feat, n_feat).astype(np.float32)

# Generate labels:
labels = np.random.choice(np.arange(n_classes), n_obs)

# Generate data:
X = np.array([mus[x] for x in labels])

# Add noise:
Eps = np.random.randn(n_obs, n_feat).astype(np.float32)
Eps = np.matmul(C, Eps.T).T 
X = X + Eps

##### Define dataloader, etc.:

In [105]:
X_torch = Variable(torch.from_numpy(X))
labels_torch = Variable(torch.from_numpy(labels))
dset = torch.utils.data.TensorDataset(X_torch, labels_torch)
data_loader = DataLoader(dset, batch_size=batch_size, shuffle=True)

##### Initialize model instance:

In [106]:
mdl = Mdl(n_inp, n_hidden, n_classes, sigma_init)
optimizer = torch.optim.Adam(mdl.parameters(), lr=lr)
loss_ce = torch.nn.CrossEntropyLoss()

##### Iterate over training epochs:

In [110]:
start_train = time.time()
for t in np.arange(n_epochs):

    for batch_idx, (curr_X, curr_labels) in enumerate(data_loader):

        print('Training epoch {} of {}, batch {} of {}...'.format(t+1, n_epochs, batch_idx+1, len(data_loader)))
        
        # Reset optimizer:
        optimizer.zero_grad()

        # Forward pass:
        output = mdl(curr_X, sigma_noise)
        
        # Compute loss:
        L_ce = loss_ce(output[0], curr_labels)
        L_sp = sparsity_loss(output[1], p_norm)
        L = beta_ce*L_ce + beta_sp*L_sp 

        # Backprop:
        L.backward()
        optimizer.step()
stop_train = time.time()
print('Training duration : {} s'.format(stop_train - start_train))

Training epoch 1 of 500, batch 1 of 5...
Training epoch 1 of 500, batch 2 of 5...
Training epoch 1 of 500, batch 3 of 5...
Training epoch 1 of 500, batch 4 of 5...
Training epoch 1 of 500, batch 5 of 5...
Training epoch 2 of 500, batch 1 of 5...
Training epoch 2 of 500, batch 2 of 5...
Training epoch 2 of 500, batch 3 of 5...
Training epoch 2 of 500, batch 4 of 5...
Training epoch 2 of 500, batch 5 of 5...
Training epoch 3 of 500, batch 1 of 5...
Training epoch 3 of 500, batch 2 of 5...
Training epoch 3 of 500, batch 3 of 5...
Training epoch 3 of 500, batch 4 of 5...
Training epoch 3 of 500, batch 5 of 5...
Training epoch 4 of 500, batch 1 of 5...
Training epoch 4 of 500, batch 2 of 5...
Training epoch 4 of 500, batch 3 of 5...
Training epoch 4 of 500, batch 4 of 5...
Training epoch 4 of 500, batch 5 of 5...
Training epoch 5 of 500, batch 1 of 5...
Training epoch 5 of 500, batch 2 of 5...
Training epoch 5 of 500, batch 3 of 5...
Training epoch 5 of 500, batch 4 of 5...
Training epoch 5